# K-means from scratch

Centroid initialization, distance calculation, assignment, centroid updates, SSE and elbow plot.

**No scikit-learn is used.** Requirements: `pandas`, `openpyxl`, `numpy`, `matplotlib`. Put `data.xlsx` beside the notebook. Change `TARGET` if the last column is not your class label.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FILE = "data.xlsx"
df = pd.read_excel(FILE).dropna().copy()
print(df.head())
print("Columns:", list(df.columns))

# K-means requires numeric features. Remove ID/label columns if present.
DROP_COLS = []  # example: ["ID", "Target"]
Xdf = df.select_dtypes(include=np.number).drop(
    columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
if Xdf.shape[1] == 0:
    raise ValueError("No numeric columns found. Encode categorical columns manually.")
X = Xdf.to_numpy(dtype=float)

# Standardize features so larger-scale columns do not dominate distance.
mu, sd = X.mean(axis=0), X.std(axis=0)
sd[sd == 0] = 1
Xs = (X - mu) / sd

def kmeans_fit(X, k, max_iter=100, seed=42):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), size=k, replace=False)].copy()
    for _ in range(max_iter):
        d = ((X[:, None, :] - centers[None, :, :])**2).sum(axis=2)
        labels = d.argmin(axis=1)
        new_centers = centers.copy()
        for j in range(k):
            if np.any(labels == j):
                new_centers[j] = X[labels == j].mean(axis=0)
        if np.allclose(new_centers, centers):
            centers = new_centers
            break
        centers = new_centers
    d = ((X[:, None, :] - centers[None, :, :])**2).sum(axis=2)
    labels = d.argmin(axis=1)
    sse = sum(((X[labels == j] - centers[j])**2).sum() for j in range(k))
    return labels, centers, float(sse)

# Elbow method: plot SSE for several K values.
ks = range(1, min(10, len(Xs)) + 1)
sses = [kmeans_fit(Xs, k)[2] for k in ks]
plt.plot(list(ks), sses, marker="o")
plt.title("Elbow method: SSE vs K")
plt.xlabel("Number of clusters (K)")
plt.ylabel("SSE / within-cluster sum of squares")
plt.xticks(list(ks)); plt.grid(True); plt.tight_layout(); plt.show()

K = 3  # choose K after inspecting elbow plot
labels, centers, sse = kmeans_fit(Xs, K)
print("SSE:", round(sse, 4))
print("Cluster sizes:", dict(pd.Series(labels).value_counts().sort_index()))
print("Standardized centroids:\\n", centers)

# Plot first two standardized features (or pad with zero if only one feature).
pts = Xs[:, :2] if Xs.shape[1] >= 2 else np.column_stack([Xs[:, 0], np.zeros(len(Xs))])
cen = centers[:, :2] if Xs.shape[1] >= 2 else np.column_stack([centers[:, 0], np.zeros(K)])
plt.figure(figsize=(7,5))
plt.scatter(pts[:,0], pts[:,1], c=labels)
plt.scatter(cen[:,0], cen[:,1], marker="X", s=180)
plt.title(f"K-means clusters (K={K})")
plt.xlabel(Xdf.columns[0])
plt.ylabel(Xdf.columns[1] if Xdf.shape[1] >= 2 else "0")
plt.tight_layout(); plt.show()

result = df.copy()
result["Cluster"] = labels
display(result.groupby("Cluster")[list(Xdf.columns)].mean(numeric_only=True))
